# Train personalized Model B on all training users

Train the full JDsearch personalized LightGBM ranker with seven baseline features plus 23 history features. All 120,174 usable training users are loaded from train_c0 through train_c3. Validation is used for early stopping only; test data is not loaded.

Check the candidates, labels, baseline features and training settings against the supplied full Model A saved under Data/Models/Baseline. These are integrity checks, not a performance comparison.

Save model_B.txt, keyed validation predictions, configuration, fingerprints and the shared evaluator under Data/Models/Advanced. Comparing A against B, computing bootstrap intervals and analysing user groups belong in a separate evaluation notebook. No comparison is performed here.


In [ ]:
# Install only missing dependencies in the current notebook environment.
import importlib.util
import subprocess
import sys
missing = [name for name in ['lightgbm', 'pyarrow', 'scikit-learn']
           if importlib.util.find_spec({'scikit-learn': 'sklearn'}.get(name, name)) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import os, json, shutil, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/bt4222_project')
    LOCAL = Path('/content/jd_local/advanced_model')
except ImportError:
    BASE = Path(os.environ['JD_BASE'])
    LOCAL = Path(os.environ['JD_LOCAL']) / 'advanced_model'

FEATURE_DIR = BASE / 'Data/Features'
SAVE_DIR = BASE / 'Data/Models/Advanced'
BASELINE_DIR = BASE / 'Data/Models/Baseline'      # Model A outputs: evaluator, predictions, fingerprints
LOCAL.mkdir(parents=True, exist_ok=True)
BASELINE_FILES = ['run_config.json', 'ranking_evaluation.py']
for name in BASELINE_FILES:
    if not (BASELINE_DIR / name).is_file():
        raise FileNotFoundError(f'Missing baseline output: {BASELINE_DIR / name}. Finish Baseline_Model first.')
base_cfg = json.loads((BASELINE_DIR / 'run_config.json').read_text())
TRAIN_TAGS = [f'train_c{i}' for i in range(4)]
if int(base_cfg.get('queries', {}).get('train', 0)) != 120174:
    raise ValueError('Expected the supplied full baseline with 120,174 training users. Check BASELINE_DIR.')
SEED = int(base_cfg['seed'])
K = 10
if base_cfg['parameters'].get('label_gain') != [0, 1, 2, 3]:
    raise ValueError('Baseline label gains differ from the planned linear gains.')
FEATURES_A = [
    'query_coverage', 'tfidf_cosine', 'brand_name_match',
    'prod_pop_ord', 'prod_pop_cart', 'prod_pop_click', 'has_meta',
]
# 23 history features from FeatureEngineering section 3 (featB_hist_<tag>), by group
FEATURE_GROUPS_B = {
    'user size':          ['n_ord', 'n_cart', 'n_click'],
    'same product':       ['prev_n_ord', 'prev_n_cart', 'prev_n_click', 'prev_days_since'],
    'candidate affinity': ['cat4_share_ord', 'cat4_share_cart', 'cat4_share_click',
                           'cat4_days_since_ord', 'cat4_days_since_cart', 'cat4_days_since_click',
                           'brand_share_ord', 'brand_share_cart', 'brand_share_click',
                           'brand_days_since_ord', 'shop_share_ord', 'deepest_match_level'],
    'past-query intent':  ['pqs_max', 'pqs_decay'],
    'collaborative filtering': ['cf_max', 'cf_score'],
}
FEATURES_B = [c for g in FEATURE_GROUPS_B.values() for c in g]
FEATURES = FEATURES_A + FEATURES_B          # Model B = Model A's features + shopper history
assert len(FEATURES_A) == 7 and len(FEATURES_B) == 23
assert len(set(FEATURES)) == len(FEATURES), 'Duplicate feature names'
if base_cfg['features'] != FEATURES_A:
    raise ValueError('Baseline feature list/order changed. Use the exact features from the baseline run.')
KEYS = ['user_id', 'wid']
print('Input:', FEATURE_DIR)
print('Output:', SAVE_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input: /content/drive/MyDrive/bt4222_project/Data/Features
Output: /content/drive/MyDrive/bt4222_project/Data/Models/Advanced


## 1 Load the same candidates and add history features

Inputs for each of train_c0..train_c3 and val: base_<tag>, featA_<tag>, and featB_hist_<tag>. The history file must contain all 23 selected columns. One-to-one joins use (user_id, wid); missing, additional and duplicate candidates are rejected.

Fingerprints compare the loaded baseline features and labels to those actually used in Model A. If they differ, stop and resolve the data change; do not bypass the checks. Changing the intended baseline inputs requires retraining both models.

Candidate order is rebuilt deterministically, with the same procedure and seed as Model A. Raw IDs and original list position never enter the model. Missing history values are allowed; infinite values are rejected.

All four training chunks are joined before the seeded shuffle, matching the supplied full Model A loader.


In [ ]:
def local_input(name):
    src = FEATURE_DIR / f'{name}.parquet'
    if not src.is_file():
        raise FileNotFoundError(f'{src} missing. Check the shared Drive link and finish FeatureEngineering first.')
    dst = LOCAL / src.name
    # Copy afresh: Drive files may have been rebuilt by a teammate.
    shutil.copy2(src, dst)
    return dst

def load_split(tags):
    tags = [tags] if isinstance(tags, str) else list(tags)
    tag = '+'.join(tags)
    f = pd.concat([pd.read_parquet(local_input(f'featA_{t}')) for t in tags], ignore_index=True)
    h = pd.concat([pd.read_parquet(local_input(f'featB_hist_{t}')) for t in tags], ignore_index=True)
    missing_a = set(KEYS + FEATURES_A) - set(f.columns)
    missing_b = set(KEYS + FEATURES_B) - set(h.columns)
    if missing_a or missing_b:
        raise ValueError(f'{tag}: missing baseline columns {missing_a}; missing history columns {missing_b}. Finish FeatureEngineering section 3.7.')
    if h[KEYS].isna().any().any() or h.duplicated(KEYS).any():
        raise ValueError(f'{tag}: null or duplicate keys in history features')
    # Model B: add the history block to Model A's features, joined by (user_id, wid)
    f = f[KEYS + FEATURES_A].merge(h[KEYS + FEATURES_B], on=KEYS, how='outer', validate='one_to_one', indicator=True)
    if not f['_merge'].eq('both').all():
        raise ValueError(f'{tag}: Model A and history feature files contain different candidates; rebuild features.')
    f = f.drop(columns='_merge')
    b = pd.concat([pd.read_parquet(local_input(f'base_{t}'), columns=KEYS + ['label']) for t in tags], ignore_index=True)
    for name, table in [('features', f), ('base', b)]:
        if table[KEYS].isna().any().any() or table.duplicated(KEYS).any():
            raise ValueError(f'{tag}: null or duplicate keys in {name}')
    absent = set(FEATURES) - set(f.columns)
    if absent:
        raise ValueError(f'{tag}: missing features {absent}')
    # Labels come exclusively from the base file; ignore any extra feature-file columns.
    d = b.merge(f[KEYS + FEATURES], on=KEYS, how='outer', validate='one_to_one', indicator=True)
    if not d['_merge'].eq('both').all():
        raise ValueError(f'{tag}: feature and label candidate sets differ; rebuild features.')
    d = d.drop(columns='_merge')
    if not d.label.isin([0, 1, 2, 3]).all():
        raise ValueError('Labels must be 0/1/2/3. Confirm the mapping in the dataset.')
    positive = d.groupby('user_id').label.transform('max').gt(0)
    excluded = d.loc[~positive, 'user_id'].nunique()
    print(tag, '| queries without positives excluded:', excluded)
    d = d.loc[positive].sort_values(KEYS).reset_index(drop=True)
    if d.empty:
        raise ValueError(f'{tag}: no usable queries')
    d['_shuffle'] = np.random.default_rng(SEED).random(len(d))
    d = d.sort_values(['user_id', '_shuffle']).drop(columns='_shuffle').reset_index(drop=True)
    d[FEATURES] = d[FEATURES].astype('float32')
    if np.isinf(d[FEATURES].to_numpy()).any():
        raise ValueError('Infinite feature values detected; NaN is allowed, infinity is not.')
    if d[['prod_pop_ord', 'prod_pop_cart', 'prod_pop_click', 'has_meta']].isna().any().any():
        raise ValueError('Popularity and has_meta must not be missing.')
    if (d[['prod_pop_ord', 'prod_pop_cart', 'prod_pop_click']] < 0).any().any():
        raise ValueError('Popularity log-counts must be nonnegative.')
    assert d.user_id.is_monotonic_increasing
    print(tag, '| rows:', len(d), '| queries:', d.user_id.nunique())
    return d

train = load_split(TRAIN_TAGS)
val = load_split('val')

# Controlled comparison: Model B must see exactly Model A's candidates, labels and Model A features.
def fingerprint(frame):
    ordered = frame.sort_values(KEYS).reset_index(drop=True)
    hashes = pd.util.hash_pandas_object(ordered, index=False).to_numpy()
    return hashlib.sha256(hashes.tobytes()).hexdigest()
for split, d in [('train', train), ('val', val)]:
    assert fingerprint(d[KEYS + ['label']]) == base_cfg['fingerprints'][f'{split}_keys_labels'], f'{split}: candidates/labels differ from Model A'
    assert fingerprint(d[KEYS + FEATURES_A]) == base_cfg['fingerprints'][f'{split}_features'], f'{split}: Model A features changed since Model A was trained'
assert train.user_id.nunique() == base_cfg['queries']['train'] == 120174
assert val.user_id.nunique() == base_cfg['queries']['val']
print('same full-training candidates, labels and Model A features as the baseline run: OK')

assert set(train.user_id).isdisjoint(set(val.user_id)), 'Train/validation user overlap'
group_train = train.groupby('user_id', sort=False).size().to_numpy(dtype=np.int32)
group_val = val.groupby('user_id', sort=False).size().to_numpy(dtype=np.int32)
assert group_train.sum() == len(train) and group_val.sum() == len(val)
display(train[FEATURES].agg(['mean', 'min', 'max']).T.assign(missing_share=train[FEATURES].isna().mean()))

print('History feature missing-value shares:')
display(pd.DataFrame({'train': train[FEATURES_B].isna().mean(), 'val': val[FEATURES_B].isna().mean()}))


train_c0+train_c1+train_c2+train_c3 | queries without positives excluded: 0
train_c0+train_c1+train_c2+train_c3 | rows: 10750112 | queries: 120174
val | queries without positives excluded: 0
val | rows: 2294519 | queries: 25787
same full-training candidates, labels and Model A features as the baseline run: OK


,mean,min,max,missing_share
query_coverage,0.721159,0.000000,1.000000,0.004544
tfidf_cosine,0.263476,0.000000,1.000000,0.004544
brand_name_match,0.049002,0.000000,1.000000,0.057788
prod_pop_ord,0.380336,0.000000,8.122372,0.000000
prod_pop_cart,0.661787,0.000000,9.982299,0.000000
prod_pop_click,0.574545,0.000000,9.060912,0.000000
has_meta,0.995456,0.000000,1.000000,0.000000
n_ord,2.692937,0.000000,7.993958,0.000000
n_cart,3.279067,0.000000,8.192847,0.000000
n_click,2.699470,0.000000,8.088562,0.000000


History feature missing-value shares:


,train,val
n_ord,0.000000,0.000000
n_cart,0.000000,0.000000
n_click,0.000000,0.000000
prev_n_ord,0.000000,0.000000
prev_n_cart,0.000000,0.000000
prev_n_click,0.000000,0.000000
prev_days_since,0.992167,0.992042
cat4_share_ord,0.087097,0.091613
cat4_share_cart,0.048189,0.049481
cat4_share_click,0.168104,0.168242


In [ ]:
print(sorted(p.name for p in FEATURE_DIR.glob('featB_hist_*')))

['featB_hist_test.parquet', 'featB_hist_train_c0.parquet', 'featB_hist_train_c1.parquet', 'featB_hist_train_c2.parquet', 'featB_hist_train_c3.parquet', 'featB_hist_train_dev.parquet', 'featB_hist_val.parquet']


## 2 Load the shared validation metric

Use the same saved evaluation function and seed as Model A. During training, its NDCG@10 selects the early-stopping iteration. This does not compare the models. Final A/B comparisons and confidence intervals will be computed in the evaluation notebook.


In [ ]:
# Reuse the evaluator saved by Model A, so both models are scored by literally the same code.
EVALUATION_CODE = (BASELINE_DIR / 'ranking_evaluation.py').read_text()
exec(EVALUATION_CODE)
evaluate_val = make_evaluator(val[KEYS], val.label.to_numpy(), k=K, seed=SEED)


## 3 Train Model B with the baseline settings

Parameters are loaded directly from Model A's saved run_config.json, rather than copied manually. Early stopping follows the same rule: 50 rounds without validation NDCG@10 improvement. The selected number of trees may differ between A and B.

Model B receives seven baseline columns plus 23 history columns. Validation NDCG@10 is monitored during training to select the early-stopping iteration. Final model comparison and test evaluation are performed in ModelComparison.ipynb.


In [ ]:
PARAMS = dict(base_cfg['parameters'])

def validation_ndcg(y_true, y_pred):
    # Only this validation dataset is passed to fit.
    if not np.array_equal(y_true, val.label.to_numpy()):
        raise ValueError('Evaluation labels do not match the validation rows')
    return 'ndcg10_tie_safe', evaluate_val(y_pred)['ndcg10'], True

model = lgb.LGBMRanker(**PARAMS)
model.fit(
    train[FEATURES], train.label.to_numpy(), group=group_train,
    eval_set=[(val[FEATURES], val.label.to_numpy())],
    eval_group=[group_val], eval_names=['validation'],
    eval_metric=validation_ndcg,
    callbacks=[lgb.early_stopping(50, first_metric_only=True), lgb.log_evaluation(25)],
)
print('Best iteration:', model.best_iteration_)
scores_b = model.predict(val[FEATURES], num_iteration=model.best_iteration_)



Training until validation scores don't improve for 50 rounds
[25]	validation's ndcg10_tie_safe: 0.400427
[50]	validation's ndcg10_tie_safe: 0.401703
[75]	validation's ndcg10_tie_safe: 0.403443
[100]	validation's ndcg10_tie_safe: 0.404873
[125]	validation's ndcg10_tie_safe: 0.405577
[150]	validation's ndcg10_tie_safe: 0.406283
[175]	validation's ndcg10_tie_safe: 0.406661
[200]	validation's ndcg10_tie_safe: 0.40685
[225]	validation's ndcg10_tie_safe: 0.407146
[250]	validation's ndcg10_tie_safe: 0.407368
[275]	validation's ndcg10_tie_safe: 0.407461
[300]	validation's ndcg10_tie_safe: 0.407715
[325]	validation's ndcg10_tie_safe: 0.408205
[350]	validation's ndcg10_tie_safe: 0.408671
[375]	validation's ndcg10_tie_safe: 0.408648
[400]	validation's ndcg10_tie_safe: 0.408774
[425]	validation's ndcg10_tie_safe: 0.408846
[450]	validation's ndcg10_tie_safe: 0.409006
[475]	validation's ndcg10_tie_safe: 0.409047
[500]	validation's ndcg10_tie_safe: 0.408918
[525]	validation's ndcg10_tie_safe: 0.40907

## 4 Save Model B for the evaluation notebook

Existing files with these four names in Data/Models/Advanced are replaced. The saved model can later score the test features without retraining. Validation predictions contain user_id, wid, label and score_B.

Only the training monitor's best validation NDCG@10 is recorded in the configuration. No A/B metrics, bootstrap intervals, group analysis or test results are calculated. Older comparison files already in the output folder are not deleted; the next notebook should use the outputs from this run.


In [ ]:
def fingerprint(frame):
    ordered = frame.sort_values(KEYS).reset_index(drop=True)
    hashes = pd.util.hash_pandas_object(ordered, index=False).to_numpy()
    return hashlib.sha256(hashes.tobytes()).hexdigest()

SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.booster_.save_model(str(LOCAL / 'model_B.txt'), num_iteration=model.best_iteration_)

# Store scores, keys and labels; the next notebook performs the comparison.
predictions = val[KEYS + ['label']].copy()
predictions['score_B'] = scores_b
predictions.to_parquet(LOCAL / 'validation_predictions.parquet', index=False)
(LOCAL / 'ranking_evaluation.py').write_text(EVALUATION_CODE)
metadata = dict(
    stage='full_validation', model='B (personalized)', training_tags=TRAIN_TAGS,
    features=FEATURES, features_baseline=FEATURES_A,
    features_history=FEATURE_GROUPS_B, parameters=PARAMS,
    seed=SEED, best_iteration=int(model.best_iteration_),
    early_stopping=dict(rounds_without_improvement=50, metric='ndcg10_tie_safe',
                        best_validation_ndcg10=float(model.best_score_['validation']['ndcg10_tie_safe'])),
    rows={'train': len(train), 'val': len(val)},
    queries={'train': len(group_train), 'val': len(group_val)},
    metric_definitions={'ndcg10': 'linear gains [0,1,2,3], top 10',
                        'mrr': 'first label >=1, full list', 'hit10': 'any label >=1, top 10'},
    no_positive_queries='excluded', tie_break='canonical keys then seeded random priority',
    versions={'python': platform.python_version(), 'numpy': np.__version__,
              'pandas': pd.__version__, 'lightgbm': lgb.__version__},
    baseline_reference=dict(directory=str(BASELINE_DIR), parameters=base_cfg['parameters'],
                            fingerprints=base_cfg['fingerprints']),
    baseline_fingerprints_verified=True,
    evaluator_sha256=hashlib.sha256(EVALUATION_CODE.encode()).hexdigest(),
    fingerprints={
        'train_keys_labels': fingerprint(train[KEYS + ['label']]),
        'val_keys_labels': fingerprint(val[KEYS + ['label']]),
        'train_features': fingerprint(train[KEYS + FEATURES]),
        'val_features': fingerprint(val[KEYS + FEATURES]),
        'train_baseline_features': fingerprint(train[KEYS + FEATURES_A]),
        'val_baseline_features': fingerprint(val[KEYS + FEATURES_A]),
    },
)
(LOCAL / 'run_config.json').write_text(json.dumps(metadata, indent=2))
for name in ['model_B.txt', 'validation_predictions.parquet', 'ranking_evaluation.py', 'run_config.json']:
    shutil.copy2(LOCAL / name, SAVE_DIR / name)
print('Saved full Model B and validation predictions to:', SAVE_DIR)
print('No model comparison or test evaluation performed.')


Saved full Model B and validation predictions to: /content/drive/MyDrive/bt4222_project/Data/Models/Advanced
No model comparison or test evaluation performed.


LightGBM API references: [LGBMRanker](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMRanker.html) and [early stopping](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.early_stopping.html).
